# Четыре тестовых сабмита: train против train+val

Оба refit проверены локально: fixed final epoch, 5698 train samples, корректные configs/scheduler/journal/source hashes, finite веса и совместимый forward.50:20 эпох, train Accuracy99.61%,15.61мин;101:21 эпоха,99.75%,30.37мин. Это **train**, не независимая метрика качества. Val Accuracy после refit отсутствует по плану.

Фиксируем четыре рецепта. Для исходных train-моделей берём **best.pt**, для refit — **final.pt**. ResNet101 безTTA, ResNet50 D4 (8 ориентаций). Ансамбль — среднее probabilities с весами0.5/0.5, а не logits/голоса.50 считается один раз,101 тоже один раз: её probabilities используются и дляsingle, и дляensemble.

| Сабмит | Данные обучения | Участники |
|---|---|---|
| resnet101_train | train4999 |101 none |
| resnet101_train_val | train+val5698 |101 none |
| ensemble_train | train4999 |101 none +50 D4 |
| ensemble_train_val | train+val5698 |101 none +50 D4 |

Test labels недоступны, поэтому здесь получаем **предсказания**, не Accuracy. CSV нужно отдельно загрузить наKaggle и вписать полученный public_score в соответствующую строкуexperiments.csv.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from src.submission_suite import CHECKPOINTS, audit_refits, run_submission_suite

RAW_DIR = PROJECT_ROOT/'data/raw'
CHECKPOINT_PATHS = dict(CHECKPOINTS)
# Если веса на Kaggle лежат в другом dataset, задайте их абсолютные пути здесь:
# CHECKPOINT_PATHS['train101'] = '/kaggle/input/.../best.pt'
# CHECKPOINT_PATHS['train50'] = '/kaggle/input/.../best.pt'
# CHECKPOINT_PATHS['refit101'] = '/kaggle/input/.../final.pt'
# CHECKPOINT_PATHS['refit50'] = '/kaggle/input/.../final.pt'

SUITE_ID = 'four_test_submissions_001'  # новый ID для нового протокола
DEVICE = 'auto'
BATCH_SIZE = 32  # при нехватке GPU памяти уменьшить до16 ДО первого запуска
NUM_WORKERS = 0
CPU_THREADS = 4
RUN_TEST = False  # включить для генерации четырех CSV, предпочтительно Kaggle GPU
RUN_AUDIT = False  # уже выполнен локально; True при наличии logs/configs/weights в проекте


## Проверка refit

Локальная проверка уже выполнена; таблица ниже сохранена вdocs/final_refit_audit.csv. Для повторной проверки нужен полный набор logs/configs и новых весов. Inference сам проверит назначение train/refit, архитектуры, порядок классов и source hashes.


In [2]:
if RUN_AUDIT:
    audit = audit_refits(PROJECT_ROOT, raw_dir=RAW_DIR, checkpoints=CHECKPOINT_PATHS)
    display(audit)
elif (PROJECT_ROOT/'docs/final_refit_audit.csv').exists():
    display(pd.read_csv(PROJECT_ROOT/'docs/final_refit_audit.csv'))
else:
    print('Local audit passed; see docs/final_refit_review.md.')
display(pd.DataFrame([{'member':name, 'checkpoint':str(path)} for name,path in CHECKPOINT_PATHS.items()]))


,exp_id,epochs,train_samples,last_train_accuracy,last_train_loss,train_time_min,checkpoint_sha256,checks
0,resnet50_final_train_val_001,20,5698,0.996139,0.012952,15.605596,0ba6befb21249ad3312ccd07d5cd597c0c9eace1ba6fdf...,passed
1,resnet101_final_train_val_001,21,5698,0.997543,0.006069,30.370029,7f75a1110cb6f3448f37523134891c1af7c2934d04b452...,passed


,member,checkpoint
0,train101,artifacts/models/resnet101_partial_domain_shif...
1,train50,artifacts/models/resnet50_partial_domain_shift...
2,refit101,artifacts/models/resnet101_final_train_val_001...
3,refit50,artifacts/models/resnet50_final_train_val_001/...


## Test inference и создание четырех CSV

Нового обучения нет. Требуются четыреcheckpoint, src и raw/test.csv, test/, classes.csv, sample_submission_example.csv. Train/val изображения дляinference не нужны. После переноса наKaggle проверьте RAW_DIR/PROJECT_ROOT/CHECKPOINT_PATHS.

Один suite ID = один фиксированный протокол. Сохраняются batch-кэши, поэтому после обрыва можно выполнить ячейку снова: готовые batches/model probabilities переиспользуются. Для resume сохранить те же weights, CSV, batch_size и SUITE_ID. Завершённый повтор не создаёт дубли в журнале. Внешняя загрузка наKaggle автоматически не выполняется.


In [3]:
if RUN_TEST:
    submissions = run_submission_suite(SUITE_ID, project_root=PROJECT_ROOT,
        raw_dir=RAW_DIR, checkpoints=CHECKPOINT_PATHS, device=DEVICE,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        cpu_threads=CPU_THREADS, resume=True)
    display(submissions)
    journal = pd.read_csv(PROJECT_ROOT/'experiments.csv', keep_default_na=False)
    display(journal[journal.submission_id.isin(submissions.submission_id)])
else:
    print('Test inference disabled. Set RUN_TEST=True to generate the four submissions.')


Test inference disabled. Set RUN_TEST=True to generate the four submissions.


## Что сохраняется

- submissions/<suite_id>_<recipe>.csv: ровно8299 строк, id,label, порядокsample_submission.
- artifacts/predictions/<suite_id>/: probabilities четырёх участников и четырёх рецептов, batch-кэши.
- artifacts/logs/<suite_id>/: configs источников вcheckpoints, SHA256/protocol/метаданные, прогресс, inference time, сведения окаждомCSV, submissions.csv.
- experiments.csv: четыре новые строки **TEST INFERENCE ONLY**, submission_id заполнен. val_accuracy, train_time_min иpublic_score пустые; времяinference хранится вmetadata, training-записи остаются отдельно.

После загрузкиCSV наKaggle внесите scores именно вновые submission-строки. Сравните отдельно эффект объединённых данных дляsingle101 идляансамбля. Не добавляйте сюда fitting/scaling/отбор моделей поtest-изображениям.
